# Operational tests: data audit and SoC method comparison

**Section 1** audits every operational test folder before any SoC method is run on it. It runs only with `RUN_AUDIT = True`, and takes about 2–3 minutes because it reads every raw file. The corrections it led to were confirmed and are stored in `soc_eval.OPERATIONAL_RUNS`; Section 1.4 lists them.

**Section 2** evaluates the run set in `RUN`, with those corrections. Run the notebook once per run. Operational runs have no reference SoC, so Section 2 reports **disagreement between methods, stability and recovery, not accuracy**.

How Section 2 sets up each run:
- **Start.** Every method starts from the OCV table at the first sample, as the server does on a fresh start.
- **Inputs.** CC, EKF and EKF + KCL integrate the offset-corrected battery current. EKF + KCL adds the gated balance update, which `kcl_correction_explained.md` concludes is harmful as a corrector.
- **States.** There is no switch-state channel, so operating states are inferred from the currents.
- **Independence.** These runs were used as checks during tuning, so they are not fully independent of the settings.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'final_report' / 'soc_eval.py').exists()) / 'final_report'))
import soc_eval as se

RUN_AUDIT = False                          # Section 1: re-run the audit of all four folders
RUN = 'LiNMC Battery Discharge Test'       # Section 2: a key of se.OPERATIONAL_RUNS
START_OFFSETS = (-0.2, -0.1, 0.1, 0.2)     # initialization spread: extra starts relative to the OCV-table start

se.apply_style()
pd.set_option('display.precision', 3)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)
models = {name: se.load_model(b['model']) for name, b in se.BATTERIES.items()}

# Section 1: audit

What the audit checks for each run:
- **Timing.** Sample rate, sessions and counter breaks. For database exports it also checks server run ids and logger stalls, where the wall clock moves on without samples, and the charge each battery passed while unrecorded.
- **Copies.** Whether any data duplicates the controlled tests or the older recordings. This compares fingerprints of the time step and voltage sequences.
- **Old wiring.** The controlled tests were wired differently: the MPPT channel carried the battery current, the battery sensor was not connected, and in the LiFePO4 test the load sensor was reversed. The audit looks for that signature.
- **Signs.** Whether each battery's voltage falls when its discharge current rises. It also checks what the server's flip handling (`detectAndCorrectFlips`) does to the run. In particular, one raw MPPT sample below −5 A latches a sign flip of every later MPPT reading.
- **Balance.** The current-balance (KCL) residual per minute and per inferred state, with the server's zeroing of disconnected batteries applied. Minutes above 2 A are flagged.
- **Usable content.** Connected slots and their voltages, and rests of 2 minutes or more.

## 1.1 Copies of controlled or older data

In [2]:
if RUN_AUDIT:
    reference_fp = np.unique(np.concatenate([se.fingerprints(sorted((se.TEST_DATA / f).glob('*.csv'), key=se._natural))
                                             for f in se.REFERENCE_FOLDERS]))
    print(f'{len(reference_fp):,} reference fingerprints')

## 1.2 Audit of each run
For each folder the cell prints:
- the file inventory;
- what the server's flip handling does to the run;
- the voltage and wiring checks;
- the residual per inferred state.

It then saves a diagnostic figure:
- currents with the signs as recorded (a disconnected battery is drawn as 0);
- slot voltages;
- the per-minute KCL residual;
- the inferred state.

In [3]:
if RUN_AUDIT:
    verdicts = []
    for folder in se.OPERATIONAL_FOLDERS:
        print(f'\n=== {folder}')
        inventory = se.file_inventory(se.TEST_DATA / folder)
        raw = se.load_run(se.TEST_DATA / folder, 'main', verbose=False, correct_flips=False)
        flips = se.flip_events(raw)
        win = se.window_median(raw)
        del raw
        audit = se.audit_signals(win, models)
        dup, _ = se.duplicate_share(se.TEST_DATA / folder, reference_fp)
        key = folder.replace(' ', '_')
        se.save_table(inventory, f'audit_{key}_files', index=False)
        display(inventory)
        print('What detectAndCorrectFlips does to this run:')
        display(flips.to_frame('value'))
        display(audit['voltages'])
        display(audit['wiring'])
        display(audit['states'])
        fig = se.plot_audit(win, audit, folder, flips['first one at (h)'])
        se.save(fig, f'audit_{key}')
        plt.show()
        verdicts.append(se.audit_verdict(folder, inventory, flips, audit, win, dup))

## 1.3 Verdict
One column per run. Residuals use the MPPT sign as recorded, so the server's flip latch is not applied.

In [4]:
if RUN_AUDIT:
    verdict = pd.concat(verdicts, axis=1)
    se.save_table(verdict, 'audit_verdict')
    display(verdict)

## 1.4 Confirmed corrections
These are taken from `soc_eval.OPERATIONAL_RUNS`.
- **`mppt_latch=False`** keeps the MPPT sign as recorded. It only changes runs where the latch triggers.
- **`files`** limits a run to a subset of its part files.
- **`slot`** and **`battery`** say which battery is evaluated, and with which model.
- **`latch_evidence`** adds the "as the server processed it" comparison in Section 2.8.

In [5]:
display(pd.DataFrame(se.OPERATIONAL_RUNS).T)

,battery,slot,mppt_latch,files,note,latch_evidence
General Discharge Test,LiNMC,main,False,"[data_20260612-153706_part_1.csv, data_20260612-153706_part_2.csv, data_20260612-153706_part_3.csv, data_20260612-15...","12 Jun, charging near full. The load sensor reads -0.22 A when off (learned by offset correction). The separate 1.4-...",NaN
General Discharge Test 2,LiNMC,main,False,None,"12 Jun, charging near full.",NaN
LiNMC Battery Discharge Test,LiNMC,main,False,None,29 Jun. MPPT flip latch disabled (one -10.24 A spike at 0.93 h would negate every later MPPT reading). 10.4 min of l...,True
LiFePO4 Battery Discharge Test,LiFePO4,alternate,False,None,22 Jun. LiFePO4 in the alternate slot; the LiNMC in the main slot is idle and not evaluated. The server restarted du...,NaN


# Section 2: evaluation of one run
## 2.1 Load and run the methods

In [6]:
run = se.OPERATIONAL_RUNS[RUN]
BATTERY, SLOT = run['battery'], run['slot']
model = models[BATTERY]
PREFIX = 'op_' + RUN.replace(' ', '_')
TITLE = f'{RUN} ({BATTERY}, {SLOT} slot)'
win = se.load_operational(RUN)
settings = se.settings(BATTERY, model, 'tuned')
soc0 = se.nearest_ocv_soc(win.V_bat[win.V_bat > se.V_CUTOFF].iloc[0], model)
est, extra = se.run_methods(win, model, settings, slot=SLOT, soc0=soc0, start_offsets=START_OFFSETS)
state = se.record_state(win, est)
rests = se.find_rests(win, se.MIN_REST_OPERATIONAL)
print(f"{TITLE}: {win.t.iloc[-1] / 3600:.2f} h recorded, {win.session.nunique()} session(s), {len(rests)} rest(s) of 2 min or more. "
      f"Start from the OCV table: {100 * soc0:.2f} %.")
print(run['note'])

LiNMC Battery Discharge Test (LiNMC, main slot): 1.48 h recorded, 1 session(s), 0 rest(s) of 2 min or more. Start from the OCV table: 93.42 %.
29 Jun. MPPT flip latch disabled (one -10.24 A spike at 0.93 h would negate every later MPPT reading). 10.4 min of logger stalls, mostly at rest; 0.24 Ah passed unrecorded (at most 0.34 pp).


## 2.2 Operating context
- **Panels:** currents (a disconnected battery is drawn as 0), the evaluated battery's voltage, the inferred state, and each method's SoC.
- **Shading:** rests of 2 minutes or more, where every current is below 0.5 A.

In [7]:
fig = se.plot_operational(win, est, rests, state, TITLE, SLOT)
se.save(fig, f'{PREFIX}_context')
plt.show()
minutes = pd.Series(state).value_counts().reindex([s for s in se.STATES if s in set(state)]) / 60
print('Minutes per inferred state:', minutes.round(1).to_dict())

Minutes per inferred state: {'rest': 13.7, 'discharging': 21.5, 'charging': 53.5, 'pass-through': 0.2}


C:\Users\Kor\AppData\Local\Temp\ipykernel_37468\2049554360.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.3 Disagreement between methods
Pairwise differences over the run. The segment table shows how each difference changes across every inferred state that lasts at least a minute, from the start of the block to its end.

In [8]:
dis = se.disagreement_table(est)
segments = se.state_segments(est, state)
se.save_table(dis, f'{PREFIX}_disagreement')
se.save_table(segments, f'{PREFIX}_segments', index=False)
display(dis)
display(segments)
fig = se.plot_disagreement(est, rests, TITLE)
se.save(fig, f'{PREFIX}_disagreement')
plt.show()

,RMS (pp),max |·| (pp),at end (pp)
OCV − CC,2.344,5.716,1.467
EKF − CC,0.434,0.873,0.873
EKF + KCL − CC,0.449,0.933,0.933
EKF + KCL − EKF,0.023,0.059,0.059


,state,start (h),end (h),minutes,OCV − CC: start,OCV − CC: end,EKF − CC: start,EKF − CC: end,EKF + KCL − CC: start,EKF + KCL − CC: end,EKF + KCL − EKF: start,EKF + KCL − EKF: end
0,rest,0.000,0.125,7.517,-0.002,-0.025,0.002,-0.007,0.002,-0.007,0.000,0.000
1,discharging,0.126,0.391,15.950,-0.721,-0.079,-0.007,-0.053,-0.007,-0.046,-0.000,0.007
2,discharging,0.394,0.423,1.733,-0.082,-0.073,-0.048,-0.043,-0.041,-0.035,0.007,0.008
3,charging,0.426,0.841,24.917,1.443,2.626,-0.040,0.326,-0.032,0.303,0.008,-0.022
4,rest,0.842,0.868,1.583,1.730,1.439,0.327,0.359,0.304,0.335,-0.022,-0.024
5,charging,0.868,0.912,2.633,1.622,1.637,0.360,0.415,0.336,0.393,-0.024,-0.022
6,discharging,0.913,0.975,3.767,1.130,0.835,0.417,0.487,0.394,0.474,-0.022,-0.013
7,charging,0.976,1.071,5.700,1.351,1.613,0.488,0.560,0.475,0.567,-0.013,0.008
8,charging,1.078,1.411,20.017,1.554,1.969,0.562,0.836,0.572,0.893,0.010,0.058
9,rest,1.412,1.436,1.433,1.676,1.618,0.836,0.845,0.894,0.905,0.058,0.059


C:\Users\Kor\AppData\Local\Temp\ipykernel_37468\122136317.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.4 Stability during rests
During a rest the current is zero, so a consistent method should hold SoC still. Each rest is labelled by the state before it, after charge or after discharge.

In [9]:
if len(rests):
    deltas = se.rest_deltas(est, rests)
    se.save_table(deltas, f'{PREFIX}_rest_deltas', index=False)
    display(rests.assign(length_min=(rests.t_end - rests.t_disc) / 60)[['rest', 'type', 'length_min', 'criterion_met', 'I_before']])
    display(se.rest_delta_table(deltas))
    fig = se.plot_operational_rests(deltas, rests, TITLE)
    se.save(fig, f'{PREFIX}_rests')
    plt.show()
else:
    print('No verified rests (every current below 0.5 A for at least 2 minutes) in this run.')

No verified rests (every current below 0.5 A for at least 2 minutes) in this run.


## 2.5 Recovery from a wrong start (initialization spread)
Each method is replayed from the OCV-table start shifted by the offsets above, clipped to 0–100%. The spread is max − min across those starts.
- CC keeps its starting spread for the whole run.
- OCV mapping has no memory, so its spread is zero by construction.
- The EKF can only close the spread where the voltage carries SoC information.

In [10]:
sp = se.spread(extra)
h = est.t.to_numpy() / 3600
k_end = np.flatnonzero(est.V > se.V_CUTOFF)[-1]
first_below = lambda x: h[np.flatnonzero(x < 1.0)[0]] if (x < 1.0).any() else np.nan
spread_table = pd.DataFrame({se.METHODS[c]: {'at start (pp)': sp[c][0], 'at end (pp)': sp[c][k_end],
                                              'first below 1 pp (h)': first_below(sp[c])} for c in se.METHODS}).T.round(4)
se.save_table(spread_table, f'{PREFIX}_init_spread')
print('Starts (%):', np.round(100 * extra['starts'], 1))
display(spread_table)
fig = se.plot_spread(est, extra, TITLE)
se.save(fig, f'{PREFIX}_init_spread')
plt.show()

Starts (%): [ 93.4  73.4  83.4 100.  100. ]


,at start (pp),at end (pp),first below 1 pp (h)
Coulomb counting,26.583,2.658e+01,NaN
OCV mapping,0.000,0.000e+00,0.000e+00
EKF,8.325,2.000e-04,3.000e-04
EKF + KCL,8.325,2.000e-04,3.000e-04


C:\Users\Kor\AppData\Local\Temp\ipykernel_37468\3751231452.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.6 Current corrections
- **Battery bias** learned by offset correction and by the gated KCL corrector.
- **KCL residual** before and after correction.
- **EKF + KCL minus EKF**, which isolates the effect of the balance update.

In [11]:
fig = se.plot_kcl(est, extra, rests, TITLE, SLOT)
se.save(fig, f'{PREFIX}_kcl')
ch = 2 if SLOT == 'main' else 3
res = {'raw': extra['res_raw'], 'after offsets': extra['res_offsets'], 'after gated KCL': extra['res_kcl']}
kcl_table = pd.DataFrame({
    'battery charge out (Ah)': [extra['Ah']['raw'], extra['Ah']['offsets'], extra['Ah']['kcl']],
    'median |KCL residual| (A)': [np.median(np.abs(r)) for r in res.values()],
    'battery bias at end (A)': [0.0, extra['bias_offsets'][ch, -1], extra['bias_kcl'][ch, -1]],
}, index=list(res)).round(4)
se.save_table(kcl_table, f'{PREFIX}_kcl')
display(kcl_table)
plt.show()

,battery charge out (Ah),median |KCL residual| (A),battery bias at end (A)
raw,-2.853,0.237,0.000
after offsets,-3.106,0.177,0.376
after gated KCL,-3.177,0.047,0.406


C:\Users\Kor\AppData\Local\Temp\ipykernel_37468\945288775.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2.7 End of run
Each method's SoC at the start and end. Where the run has a rest of at least 5 minutes, every method is also compared with the OCV table at the end of the last such rest.
That comparison is a *shared-model consistency check*, not an accuracy check, because the same model's OCV table is inside OCV mapping and the EKF.

In [12]:
end = se.end_of_run(est, rests, model)
se.save_table(end, f'{PREFIX}_end_of_run')
display(end)
d = se.pairwise(est).dropna()
summary = pd.Series({
    'battery': BATTERY, 'slot': SLOT, 'recorded (h)': win.t.iloc[-1] / 3600, 'rests >= 2 min': len(rests),
    'OCV − CC RMS (pp)': np.sqrt((d['OCV − CC'] ** 2).mean()),
    'EKF − CC max |·| (pp)': d['EKF − CC'].abs().max(), 'EKF − CC at end (pp)': d['EKF − CC'].iloc[-1],
    'EKF + KCL − EKF max per-minute |·| (pp)': d['EKF + KCL − EKF'].groupby((est.t[d.index] // 60).astype(int)).median().abs().max(),
    'EKF spread first below 1 pp (h)': spread_table.loc['EKF', 'first below 1 pp (h)'],
    'EKF − OCV table at last 5-min rest (pp)': end.iloc[:, -1].get('EKF', np.nan) if end.shape[1] > 3 else np.nan,
}, name=RUN).map(lambda v: round(float(v), 3) if isinstance(v, (int, float, np.floating)) else v)
se.save_table(summary.to_frame(), f'{PREFIX}_summary')
display(summary.to_frame())

,start (%),end (%),change (pp)
Coulomb counting,93.417,97.765,4.348
OCV mapping,93.417,99.232,5.815
EKF,93.417,98.638,5.222
EKF + KCL,93.417,98.698,5.281


,LiNMC Battery Discharge Test
battery,LiNMC
slot,main
recorded (h),1.481
rests >= 2 min,0.0
OCV − CC RMS (pp),2.344
EKF − CC max |·| (pp),0.873
EKF − CC at end (pp),0.873
EKF + KCL − EKF max per-minute |·| (pp),0.059
EKF spread first below 1 pp (h),0.0
EKF − OCV table at last 5-min rest (pp),NaN


## 2.8 As the server processed it (KCL evidence)
This runs only for runs marked `latch_evidence`, which is the 29 Jun run. The same run is processed with the server's MPPT flip latch on, so one raw spike negates every later MPPT reading. Offset correction is unaffected. The gated KCL update absorbs the error into its bias estimates.

In [13]:
if run.get('latch_evidence'):
    win_srv = se.load_operational(RUN, mppt_latch=True)
    est_srv, extra_srv = se.run_methods(win_srv, model, settings, slot=SLOT, soc0=soc0)
    rows = {}
    for name, (e, x) in {'MPPT sign as recorded': (est, extra),
                         'as the server processed it (flip latch on)': (est_srv, extra_srv)}.items():
        minute = (e.t // 60).astype(int).to_numpy()
        diff = pd.Series(100 * (e.soc_ekf_kcl - e.soc_ekf).to_numpy()).groupby(minute).median()
        res = pd.Series(x['res_raw']).where((e.V > se.V_CUTOFF).to_numpy()).groupby(minute).median()
        rows[name] = {
            'battery charge out, offsets (Ah)': x['Ah']['offsets'],
            'battery charge out, gated KCL (Ah)': x['Ah']['kcl'],
            'largest |battery bias|, gated KCL (A)': np.abs(x['bias_kcl'][ch]).max(),
            'largest |MPPT bias|, gated KCL (A)': np.abs(x['bias_kcl'][1]).max(),
            'EKF + KCL − EKF at end (pp)': diff.iloc[-1],
            'largest per-minute |EKF + KCL − EKF| (pp)': diff.abs().max(),
            'voltage innovation RMS, EKF (mV)': x['innov_rms_mV']['soc_ekf'],
            'voltage innovation RMS, EKF + KCL (mV)': x['innov_rms_mV']['soc_ekf_kcl'],
            'minutes with |KCL residual| > 2 A': int((res.abs() > se.FAULT_RESIDUAL).sum()),
        }
    evidence = pd.DataFrame(rows).round(3)
    se.save_table(evidence, f'{PREFIX}_latch_evidence')
    display(evidence)
    fig = se.plot_kcl(est_srv, extra_srv, rests, f'{RUN}, as the server processed it', SLOT)
    se.save(fig, f'{PREFIX}_kcl_server_latch')
    plt.show()
else:
    print('Not applicable to this run.')

,MPPT sign as recorded,as the server processed it (flip latch on)
"battery charge out, offsets (Ah)",-3.106,-3.106
"battery charge out, gated KCL (Ah)",-3.177,-2.456
"largest |battery bias|, gated KCL (A)",0.488,2.617
"largest |MPPT bias|, gated KCL (A)",0.095,0.610
EKF + KCL − EKF at end (pp),0.059,-0.508
largest per-minute |EKF + KCL − EKF| (pp),0.059,0.514
"voltage innovation RMS, EKF (mV)",15.025,15.025
"voltage innovation RMS, EKF + KCL (mV)",14.482,22.968
minutes with |KCL residual| > 2 A,0.000,18.000


C:\Users\Kor\AppData\Local\Temp\ipykernel_37468\641195734.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
